# Laya — Toplumsal Katkı Türkçe (`laya-multilingual` → `laya-katki-tr-v1`)

Kurum haberlerinde karar soruları: `katki` (toplumsal katkı mı), `kategori`, `faaliyet`, `bolge` ve bölgeye
göre `il_*`. Etiketler Jev'den (typesafe/jev-1.13) damıtılmış yumuşak olasılıklar + koordinatörlük onayları.
Bölünme kaynak sitesine göre. Zero-shot ile fine-tuned modeli aynı `test`te karşılaştırır, `katki` eşiğini
calib'de seçer ve modelle birlikte kaydeder.

Resmî Laya notebook'undan uyarlanmıştır (`NandhaKishorM/laya`, Apache-2.0).

### Kaggle ayarları
* **Accelerator:** `GPU T4 x2` · **Internet:** `On` · **Input:** `laya-katki-tr` veri seti

In [ ]:
import os, torch
n_gpu = torch.cuda.device_count()
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")
assert n_gpu >= 1, "GPU yok: Notebook options -> Accelerator -> GPU T4 x2 seç"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip install -q "laya==0.3.20" "transformers>=4.48.0" safetensors huggingface_hub
import laya, transformers
print("laya", laya.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)

## 1. Ayarlar ve veri

In [ ]:
import glob, json, time
DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0])
WORK = "/kaggle/working"
MODEL_NAME = "laya-katki-tr-v1"
OUTPUT_DIR = f"{WORK}/{MODEL_NAME}"
REPORT_DIR = f"{WORK}/reports"
os.makedirs(REPORT_DIR, exist_ok=True)

MODEL_ID = "convaiinnovations/laya-multilingual"
MAX_LEN = 768           # CPU çıkarım hızı için; bölge sorusunda bile ~240 token haber metni kalır
HEAD_MAX_LEN = 544      # bölge sorusu (10 seçenek, il listeleri) ~520 token
EPOCHS = 3
EVAL_SPLITS = ("test",)

def read_split(name):
    with open(f"{DATA_DIR}/{name}.jsonl", encoding="utf-8") as f:
        return [json.loads(l) for l in f]

splits = {s: read_split(s) for s in ("train", "calib") + EVAL_SPLITS}
QUESTIONS = json.load(open(f"{DATA_DIR}/questions_tr.json", encoding="utf-8"))
print("DATA_DIR:", DATA_DIR, {s: len(v) for s, v in splits.items()}, "| sorular:", list(QUESTIONS))

In [ ]:
%%writefile /kaggle/working/metrics.py
"""Decision metrics shared by the local eval scripts and the Kaggle notebook (copied in verbatim).

Only numpy is required so the file can be `%%writefile`d into a notebook.
Uyarlama: chatbot/laya_ft/eval/metrics.py; kabul ölçütleri toplumsal katkı damıtması için (öğretmen: Jev),
il soruları bölgeye göre ayrı (il_marmara, ...) olduğundan "il" toplu doğruluğu ayrıca hesaplanır.
"""
from __future__ import annotations

import numpy as np

# Recall targets need precision floors: a model that answers "true" to everything
# (as zero-shot laya-multilingual does on Turkish) would otherwise pass them.
ACCEPTANCE = {
    "katki.accuracy": (">=", 0.92),
    "kategori.accuracy": (">=", 0.80),
    "bolge.accuracy": (">=", 0.85),
    "il.accuracy": (">=", 0.85),
    "overall.ece": ("<=", 0.08),
}


def ece(conf: np.ndarray, correct: np.ndarray, bins: int = 15) -> float:
    edges = np.linspace(0, 1, bins + 1)
    total, err = len(conf), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            err += m.sum() / total * abs(conf[m].mean() - correct[m].mean())
    return float(err)


def _prf(pred: np.ndarray, gold: np.ndarray) -> dict:
    tp = int((pred & gold).sum()); fp = int((pred & ~gold).sum()); fn = int((~pred & gold).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return {"precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0,
            "support_pos": int(gold.sum()), "predicted_pos": int(pred.sum())}


# F-beta used when choosing decision thresholds on calib: recall-weighted where a miss is costly
THRESHOLD_BETA = {"katki": 1.0}
MIN_PRECISION = 0.80


def choose_thresholds(preds: list[dict], golds: list[dict], questions: dict) -> dict:
    """Per noul question, the threshold maximising F-beta with precision >= MIN_PRECISION (if reachable)."""
    out = {}
    for qid, q in questions.items():
        if q["type"] != "noul":
            continue
        keep = [i for i, g in enumerate(golds) if qid in g]
        if not keep:
            continue
        p = np.array([preds[i][qid]["noul"] for i in keep], dtype=float)
        gold = np.array([golds[i][qid]["label"] == "true" for i in keep])
        beta = THRESHOLD_BETA.get(qid, 1.0)
        best, best_ok = (0.5, -1.0), (None, -1.0)
        for t in np.round(np.arange(0.05, 0.96, 0.05), 2):
            m = _prf(p >= t, gold)
            fb = ((1 + beta**2) * m["precision"] * m["recall"] / (beta**2 * m["precision"] + m["recall"])
                  if m["precision"] + m["recall"] else 0.0)
            if fb > best[1]:
                best = (float(t), fb)
            if m["precision"] >= MIN_PRECISION and fb > best_ok[1]:
                best_ok = (float(t), fb)
        out[qid] = best_ok[0] if best_ok[0] is not None else best[0]
    return out


def evaluate(preds: list[dict], golds: list[dict], questions: dict, thresholds: dict | None = None) -> dict:
    """preds[i][qid] is a Laya answer dict, golds[i][qid] a typed-decisions gold dict (may be absent)."""
    out, all_conf, all_corr = {}, [], []
    for qid, q in questions.items():
        t = q["type"]
        keep = [i for i, g in enumerate(golds) if qid in g]  # gold may omit disputed questions
        qp, qg = [preds[i] for i in keep], [golds[i] for i in keep]
        if not keep:
            continue
        if t == "choice":
            keys = list(q["criteria"].keys())
            P = np.array([[p[qid]["probabilities"].get(k, 0.0) for k in keys] for p in qp])
            P = P / np.clip(P.sum(1, keepdims=True), 1e-9, None)
            G = np.array([[g[qid]["probabilities"].get(k, 0.0) for k in keys] for g in qg])
            pi, gi = P.argmax(1), np.array([keys.index(g[qid]["label"]) for g in qg])
            corr = (pi == gi).astype(float)
            conf = P.max(1)
            conf_m = np.zeros((len(keys), len(keys)), dtype=int)
            for a, b in zip(gi, pi):
                conf_m[a, b] += 1
            per_class = {k: float(conf_m[i, i] / conf_m[i].sum()) if conf_m[i].sum() else None
                         for i, k in enumerate(keys)}
            out[qid] = {"accuracy": float(corr.mean()), "brier": float(((P - G) ** 2).sum(1).mean()),
                        "ece": ece(conf, corr), "per_class_recall": per_class,
                        "confusion": {"labels": keys, "matrix": conf_m.tolist()}}
        elif t == "noul":
            p = np.array([x[qid]["noul"] for x in qp], dtype=float)
            gp = np.array([g[qid]["noul"] for g in qg], dtype=float)
            thr = (thresholds or {}).get(qid, 0.5)
            pred, gold = p >= thr, np.array([g[qid]["label"] == "true" for g in qg])
            conf = np.maximum(p, 1 - p)
            # calibration is about the probability itself, so ECE uses the 0.5 decision regardless of thr
            cal_corr = ((p >= 0.5) == gold).astype(float)
            out[qid] = {"accuracy": float((pred == gold).mean()), "brier": float((2 * (p - gp) ** 2).mean()),
                        "ece": ece(conf, cal_corr), "threshold": thr, **_prf(pred, gold)}
            corr = cal_corr
        else:  # score
            n = len(q["criteria"])
            P = np.array([[x[qid]["probabilities"].get(str(i), 0.0) for i in range(n)] for x in qp])
            P = P / np.clip(P.sum(1, keepdims=True), 1e-9, None)
            gi = np.array([int(g[qid]["label"]) for g in qg])
            pi = P.argmax(1)
            corr = (pi == gi).astype(float)
            conf = P.max(1)
            exp_p = (P * np.arange(n)).sum(1)
            exp_g = np.array([g[qid]["score"] for g in qg], dtype=float)
            out[qid] = {"accuracy": float(corr.mean()), "within_1": float((abs(pi - gi) <= 1).mean()),
                        "mae_expected": float(abs(exp_p - exp_g).mean()), "ece": ece(conf, corr)}
        out[qid]["n"] = len(keep)
        all_conf.append(conf); all_corr.append(corr)
    # il soruları bölgeye göre ayrı: toplu "il" doğruluğu karar sayısıyla ağırlıklı
    il_q = [q for q in out if q.startswith("il_")]
    if il_q:
        n_il = sum(out[q]["n"] for q in il_q)
        out["il"] = {"accuracy": sum(out[q]["accuracy"] * out[q]["n"] for q in il_q) / n_il, "n": n_il,
                     "ece": sum(out[q]["ece"] * out[q]["n"] for q in il_q) / n_il}
    c, k = np.concatenate(all_conf), np.concatenate(all_corr)
    # overall accuracy/ECE are threshold-free (argmax / 0.5), so they compare models, not operating points
    out["overall"] = {"accuracy": float(k.mean()), "ece": ece(c, k), "n_states": len(preds),
                      "n_decisions": int(len(k))}
    out["acceptance"] = acceptance(out)
    return out


def acceptance(m: dict) -> dict:
    res = {}
    for key, (op, thr) in ACCEPTANCE.items():
        q, metric = key.split(".")
        v = m.get(q, {}).get(metric)
        ok = v is not None and (v >= thr if op == ">=" else v <= thr)
        res[key] = {"value": v, "target": f"{op} {thr}", "pass": bool(ok)}
    return res


def summary_rows(m: dict) -> list[tuple[str, str]]:
    rows = []
    for qid, v in m.items():
        # per-question entries only; skip overall/acceptance and extras such as "meta"
        if qid == "overall" or not isinstance(v, dict) or "accuracy" not in v:
            continue
        extra = f" recall={v['recall']:.3f} prec={v['precision']:.3f}" if "recall" in v else ""
        rows.append((qid, f"acc={v['accuracy']:.3f} ece={v['ece']:.3f}{extra}"))
    rows.append(("overall", f"acc={m['overall']['accuracy']:.3f} ece={m['overall']['ece']:.3f}"))
    return rows

In [ ]:
%%writefile /kaggle/working/prep.py
"""Turn typed-decisions rows into Laya training items (shared by the notebook and the local smoke test).

Same logic as `build_training_item` in the official Laya notebook, parameterised on token budgets.
"""
from __future__ import annotations

import json

from laya.common import QTYPES, build_sequence, render_options


def build_training_item(tok, state, q, gold_q, max_len, head_max_len):
    t, crit = q["type"], q.get("criteria", {})
    if t == "choice":
        target = [gold_q["probabilities"].get(k, 0.0) for k in crit.keys()]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    else:
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(len(crit))]
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    qq = {"t": t, "ins": q["instructions"], "crit": crit}
    seq, markers = build_sequence(tok, state, qq, max_len, head_max_len)
    if len(markers) != len(render_options(qq)):
        return None
    return {"ids": seq, "markers": markers, "qtype": QTYPES[t], "target": target,
            "label": target.index(max(target))}


def build_items(tok, rows, questions, max_len, head_max_len):
    items, dropped = [], 0
    for row in rows:
        state, gold = json.loads(row["state"]), json.loads(row["gold"])
        for qid, q in questions.items():
            if qid not in gold:  # question disputed between generator and labeller for this message
                continue
            it = build_training_item(tok, state, q, gold[qid], max_len, head_max_len)
            if it:
                items.append(it)
            else:
                dropped += 1
    return items, dropped

## 2. Ön işleme (resmî `build_sequence`)

In [ ]:
import sys
sys.path.insert(0, WORK)
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from prep import build_items

model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))

for name in ("train", "calib"):
    items, dropped = build_items(tok, splits[name], QUESTIONS, MAX_LEN, HEAD_MAX_LEN)
    torch.save(items, f"{WORK}/{name}_items.pt")
    print(f"{name}: {len(items)} items ({dropped} dropped for truncated options)")

## 3. Zero-shot baseline (aynı test, GPU)

In [ ]:
from metrics import choose_thresholds, evaluate, summary_rows

def predict_rows(agent, rows, questions, batch=16):
    preds = []
    for i in range(0, len(rows), batch):
        chunk = rows[i:i + batch]
        res = agent.predict_batch([json.loads(r["state"]) for r in chunk], questions,
                                  max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN)
        preds.extend(r["answers"] for r in res)
    return preds

def run_eval(agent, rows, questions, thresholds=None, save_as=None):
    t0 = time.time()
    preds = predict_rows(agent, rows, questions)
    m = evaluate(preds, [json.loads(r["gold"]) for r in rows], questions, thresholds)
    m["meta"] = {"n": len(rows), "seconds": round(time.time() - t0, 1)}
    if save_as:  # kayıt bazlı cevaplar: yerelde Jev ile karşılaştırma ve hata incelemesi için
        with open(f"{REPORT_DIR}/{save_as}", "w", encoding="utf-8") as f:
            for r, p in zip(rows, preds):
                f.write(json.dumps({"id": r["id"], "answers": p}, ensure_ascii=False) + "\n")
    return m

def show(name, m):
    print(f"--- {name} ---")
    for q, line in summary_rows(m):
        print(f"{q:>22}: {line}")
    for k, v in m["acceptance"].items():
        val = "yok" if v["value"] is None else f"{v['value']:.3f}"
        print(f"  {'PASS' if v['pass'] else 'FAIL'} {k} = {val} (hedef {v['target']})")

base_agent = laya.load(MODEL_ID, device="cuda")
for split in EVAL_SPLITS:
    m = run_eval(base_agent, splits[split], QUESTIONS)
    m["meta"].update(model="zero-shot", split=split)
    json.dump(m, open(f"{REPORT_DIR}/zero_shot_{split}.json", "w"), indent=2, ensure_ascii=False)
    show(f"zero-shot [{split}]", m)
del base_agent
torch.cuda.empty_cache()

## 4. Fine-tune (DDP, `torchrun`)

In [ ]:
%%writefile /kaggle/working/train_ddp.py
# Adapted from the official Laya Kaggle notebook
# (NandhaKishorM/laya notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb, Apache-2.0).
# Changes: calibration uses our own held-out `calib` split instead of carving it out of train,
# configurable max_len/head_max_len (Turkish options need a 384-token head budget), works with
# 1 or 2 GPUs, and records training metadata in rl_agent_config.json.
#
#   torchrun --standalone --nproc_per_node=<gpus> train_ddp.py MODEL_DIR OUTPUT_DIR TRAIN_PT CALIB_PT \
#       [--epochs 4] [--max-len 512] [--head-max-len 384] [--model-name laya-butik-tr]
import argparse
import json
import os
import random
import time

import torch
import torch.distributed as dist
from safetensors.torch import load_file, save_file
from torch.nn.parallel import DistributedDataParallel as DDP
from transformers import AutoTokenizer

from laya.common import build_model, proper_reward


def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items]),
    }


def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, : len(z)] = torch.tensor(z)
        T[i, : len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss

    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("model_dir")
    ap.add_argument("output_dir")
    ap.add_argument("train_pt")
    ap.add_argument("calib_pt")
    ap.add_argument("--epochs", type=int, default=4)
    ap.add_argument("--max-len", type=int, default=512)
    ap.add_argument("--head-max-len", type=int, default=384)
    ap.add_argument("--model-name", default="laya-butik-tr")
    ap.add_argument("--max-steps", type=int, default=None, help="smoke test: stop after N micro-batches")
    args = ap.parse_args()

    # CUDA on Kaggle; without it (local smoke test) fall back to gloo on CPU without mixed precision
    cuda = torch.cuda.is_available()
    dist.init_process_group("nccl" if cuda else "gloo")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    if cuda:
        torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank) if cuda else torch.device("cpu")

    with open(os.path.join(args.model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = args.max_len
    cfg["head_max_len"] = args.head_max_len

    tok = AutoTokenizer.from_pretrained(os.path.join(args.model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(args.model_dir, "encoder"))
    model.load_state_dict(load_file(os.path.join(args.model_dir, "model.safetensors")), strict=True)
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()
    ddp_model = DDP(model, device_ids=[local_rank] if cuda else None, find_unused_parameters=True)

    train_items = torch.load(args.train_pt, weights_only=False)
    calib_items = torch.load(args.calib_pt, weights_only=False)
    if args.max_steps:
        calib_items = calib_items[:32]
    my_items = train_items[rank::world_size]

    EPOCHS = args.epochs
    MICRO_BATCH = 8
    GRAD_ACCUM = max(1, 8 // world_size)  # effective batch 64 sequences on 1 or 2 GPUs
    GROUP_SIZE = 4
    LR_ENCODER = 2.5e-5
    LR_HEAD = 1.0e-4
    SIGMA_START = 0.4
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW(
        [{"params": enc_params, "lr": LR_ENCODER}, {"params": head_params, "lr": LR_HEAD}], weight_decay=0.01
    )
    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=cuda)

    if rank == 0:
        print(f"Training on {world_size} GPU(s): {len(train_items)} train items, {len(calib_items)} calib items, "
              f"{len(my_items)} per rank, {EPOCHS} epochs, ~{total_updates} updates")
    t0 = time.time()

    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches, accum_step = 0.0, 0, 0
        optimizer.zero_grad(set_to_none=True)
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress

        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx : b_idx + MICRO_BATCH]
            if not chunk:
                continue
            batch = collate_train_batch(chunk, tok.pad_token_id)
            with torch.autocast(device.type, dtype=torch.float16, enabled=cuda):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device),
                )
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)

            # Sample G noisy logit distributions with zero-mean projection
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)

            # Proper scoring reward
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)

            # Policy gradient + soft cross-entropy guidance
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma**2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()

            scaler.scale(loss).backward()
            accum_step += 1
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            if args.max_steps and n_batches >= args.max_steps:
                if rank == 0:
                    print(f"  smoke step {n_batches}: loss {loss.item() * GRAD_ACCUM:.4f} reward {r.mean().item():.3f}")
                break
            if rank == 0 and n_batches % 50 == 0:
                print(f"  Epoch {epoch + 1}/{EPOCHS} | Step {n_batches} | Loss {loss.item() * GRAD_ACCUM:.4f} | "
                      f"Reward {r.mean().item():.3f} | LR {scheduler.get_last_lr()[0]:.2e}", flush=True)

        if rank == 0:
            print(f"=== Epoch {epoch + 1}/{EPOCHS} done at {time.time() - t0:.0f}s | "
                  f"avg loss {epoch_loss / max(1, n_batches):.4f} ===", flush=True)
        dist.barrier()

        # Rolling checkpoint so a Kaggle timeout does not lose finished epochs
        if rank == 0:
            ckpt_dir = os.path.join(args.output_dir, "checkpoint_latest")
            os.makedirs(ckpt_dir, exist_ok=True)
            save_file({k: v.half().contiguous().cpu() for k, v in model.state_dict().items()},
                      os.path.join(ckpt_dir, "model.safetensors"))
            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump({"epoch": epoch + 1, "total_epochs": EPOCHS,
                           "avg_loss": epoch_loss / max(1, n_batches)}, f, indent=2)

    dist.barrier()

    if rank == 0:
        print("\nFitting calibration temperatures on the held-out calib split...")
        del optimizer, scaler, scheduler
        if cuda:
            torch.cuda.empty_cache()
        model.eval()
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx : c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with torch.autocast(device.type, dtype=torch.float16, enabled=cuda):
                    l_sub, _ = model(
                        cb["input_ids"].to(device),
                        cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device),
                        cb["marker_mask"].to(device),
                        cb["qtype"].to(device),
                    )
                l_np = l_sub.float().cpu().numpy()
                for r_i, it in enumerate(c_chunk):
                    calib_preds.append((it["qtype"], l_np[r_i, : len(it["markers"])], it["target"]))

        fitted = [1.2, 1.2, 1.2]  # QTYPES order: choice, score, noul
        try:
            for qt in range(3):
                sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
                if sel:
                    fitted[qt] = fit_one_temp(sel)
            print("Fitted temperatures (choice, score, noul):", [round(t, 3) for t in fitted])
        except Exception as e:  # noqa: BLE001
            print("Temperature fitting fallback:", e)

        os.makedirs(args.output_dir, exist_ok=True)
        save_file({k: v.half().contiguous().cpu() for k, v in model.state_dict().items()},
                  os.path.join(args.output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(args.output_dir, "encoder"))
        tok.save_pretrained(os.path.join(args.output_dir, "tokenizer"))
        cfg["fine_tuned"] = True
        cfg["model_name"] = args.model_name
        cfg["temperature"] = fitted
        cfg.pop("temperature_by_options", None)
        cfg["training"] = {"base": "convaiinnovations/laya-multilingual", "epochs": EPOCHS,
                           "train_items": len(train_items), "calib_items": len(calib_items),
                           "world_size": world_size, "minutes": round((time.time() - t0) / 60, 1)}
        with open(os.path.join(args.output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Saved fine-tuned model to {args.output_dir}")

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
cmd = (f"torchrun --standalone --nproc_per_node={n_gpu} {WORK}/train_ddp.py {model_dir} {OUTPUT_DIR} "
       f"{WORK}/train_items.pt {WORK}/calib_items.pt --epochs {EPOCHS} --max-len {MAX_LEN} "
       f"--head-max-len {HEAD_MAX_LEN} --model-name {MODEL_NAME}")
print(cmd)
t0 = time.time()
!{cmd}
print(f"Training wall time: {(time.time() - t0) / 60:.1f} min")

## 5. Fine-tuned değerlendirme

In [ ]:
ft_agent = laya.load(OUTPUT_DIR, device="cuda")

# karar eşiği calib'de seçilir (test'te asla), modelle birlikte kaydedilir
calib_preds = predict_rows(ft_agent, splits["calib"], QUESTIONS)
THRESHOLDS = choose_thresholds(calib_preds, [json.loads(r["gold"]) for r in splits["calib"]], QUESTIONS)
print("calib'de seçilen eşikler:", THRESHOLDS)
cfg_path = f"{OUTPUT_DIR}/rl_agent_config.json"
cfg = json.load(open(cfg_path)); cfg["decision_thresholds"] = THRESHOLDS
json.dump(cfg, open(cfg_path, "w"), indent=2)

results = {}
for split in EVAL_SPLITS:
    m = run_eval(ft_agent, splits[split], QUESTIONS, THRESHOLDS, save_as=f"preds_{split}.jsonl")
    m["meta"].update(model=MODEL_NAME, split=split, thresholds=THRESHOLDS)
    json.dump(m, open(f"{REPORT_DIR}/finetuned_{split}.json", "w"), indent=2, ensure_ascii=False)
    show(f"fine-tuned [{split}]", m)
    results[split] = m

compare_rows = [("katki", "accuracy"), ("katki", "precision"), ("katki", "recall"), ("kategori", "accuracy"),
                ("faaliyet", "accuracy"), ("bolge", "accuracy"), ("il", "accuracy"), ("overall", "accuracy"),
                ("overall", "ece")]
for split in EVAL_SPLITS:
    zs, ft = json.load(open(f"{REPORT_DIR}/zero_shot_{split}.json")), results[split]
    print(f"\n[{split}] metric                 zero-shot   fine-tuned")
    for q, k in compare_rows:
        z, f = zs.get(q, {}).get(k), ft.get(q, {}).get(k)
        print(f"{q + '.' + k:<30}{(z if z is not None else float('nan')):>10.3f}{(f if f is not None else float('nan')):>13.3f}")

## 6. Paketle

In [ ]:
import shutil
shutil.rmtree(f"{OUTPUT_DIR}/checkpoint_latest", ignore_errors=True)
shutil.copytree(REPORT_DIR, f"{OUTPUT_DIR}/reports", dirs_exist_ok=True)
for tmp in ("train_items.pt", "calib_items.pt"):
    if os.path.exists(f"{WORK}/{tmp}"):
        os.remove(f"{WORK}/{tmp}")
print("Hazır:", OUTPUT_DIR)